## C++ Posterior function performance & correctness evaluation

***

In [1]:
import unittest
import numpy as np
import pandas as pd
import ctypes
from customProphet import *

class TestLogPosterior(unittest.TestCase):
    def setUp(self):
        # Load the shared library
        self.lib = ctypes.CDLL('./libminus_log_posterior.so')
        self.lib.minus_log_posterior.argtypes = [
            np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
            ctypes.c_size_t,
            np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
            ctypes.c_size_t,
            np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
            ctypes.c_size_t,
            ctypes.c_double,
            np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
            ctypes.c_size_t,
            ctypes.c_double,
            ctypes.c_double,
            ctypes.c_double,
            ctypes.c_double,
            ctypes.c_double
        ]
        self.lib.minus_log_posterior.restype = ctypes.c_double

        # Load data
        df = pd.read_csv('peyton_manning.csv')

        # Instantiate & initialize a model
        self.model = CustomProphet()
        self.model.y = df['y'].values
        if df['ds'].dtype != 'datetime64[ns]':
            self.model.ds = pd.to_datetime(df['ds'])
        else:
            self.model.ds = df['ds']

        self.model.t_scaled = np.array((self.model.ds - self.model.ds.min()) / (self.model.ds.max() - self.model.ds.min()))
        self.model.T = df.shape[0]

        self.model.scale_period = (self.model.ds.max() - self.model.ds.min()).days
        self.model._normalize_y()
        self.model._generate_change_points()

        self.params = np.ones((47,))
    
    def tearDown(self):
        # This method is called after each test
        print('Test passed: posterior function correctly implemented')

    def test_minus_log_posterior(self):
        # Convert data to ctypes
        params_ctypes = np.ascontiguousarray(self.params, dtype=np.float64)
        t_scaled_ctypes = np.ascontiguousarray(self.model.t_scaled, dtype=np.float64)
        change_points_ctypes = np.ascontiguousarray(self.model.change_points, dtype=np.float64)
        normalized_y_ctypes = np.ascontiguousarray(self.model.normalized_y, dtype=np.float64)

        # Call the Python method
        mpl_python = self.model._minus_log_posterior(self.params)

        # Call the C++ function
        mlp_cpp = self.lib.minus_log_posterior(
            params_ctypes, len(params_ctypes),
            t_scaled_ctypes, len(t_scaled_ctypes),
            change_points_ctypes, len(change_points_ctypes),
            self.model.scale_period,
            normalized_y_ctypes, len(normalized_y_ctypes),
            self.model.sigma_obs,
            self.model.sigma_k,
            self.model.sigma_m,
            self.model.sigma,
            self.model.tau
        )

        # Assert that the values are close
        self.assertAlmostEqual(mpl_python, mlp_cpp, places=5, msg="The C++ and Python minus log posterior values do not match.")

# Run in a jupyter notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestLogPosterior)

# Run the test suite
unittest.TextTestRunner(verbosity=2).run(suite)

# Run in a script
#if __name__ == '__main__':
 #   unittest.main()

test_minus_log_posterior (__main__.TestLogPosterior.test_minus_log_posterior) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.293s

OK


Test passed: posterior function correctly implemented


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [2]:
import numpy as np
import pandas as pd
import ctypes
from customProphet import *

# Load the shared library
lib = ctypes.CDLL('./libminus_log_posterior.so')

# Define argument and return types
lib.minus_log_posterior.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_size_t,
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_size_t,
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_size_t,
    ctypes.c_double,
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_size_t,
    ctypes.c_double,
    ctypes.c_double,
    ctypes.c_double,
    ctypes.c_double,
    ctypes.c_double
]
lib.minus_log_posterior.restype = ctypes.c_double

# Load data
df = pd.read_csv('peyton_manning.csv')

# Instantiate & initialize a model
model = CustomProphet()
model.y = df['y'].values
if df['ds'].dtype != 'datetime64[ns]':
    model.ds = pd.to_datetime(df['ds'])
else:
    model.ds = df['ds']

model.t_scaled = np.array((model.ds - model.ds.min()) / (model.ds.max() - model.ds.min()))
model.T = df.shape[0]

model.scale_period = (model.ds.max() - model.ds.min()).days
model._normalize_y()
model._generate_change_points()

params = np.ones((47,))

# Convert data to ctypes
params_ctypes = np.ascontiguousarray(params, dtype=np.float64)
t_scaled_ctypes = np.ascontiguousarray(model.t_scaled, dtype=np.float64)
change_points_ctypes = np.ascontiguousarray(model.change_points, dtype=np.float64)
normalized_y_ctypes = np.ascontiguousarray(model.normalized_y, dtype=np.float64)

mpl = model._minus_log_posterior(params)

# Call the C++ function
mlp_cpp = lib.minus_log_posterior(
    params_ctypes, len(params_ctypes),
    t_scaled_ctypes, len(t_scaled_ctypes),
    change_points_ctypes, len(change_points_ctypes),
    model.scale_period,
    normalized_y_ctypes, len(normalized_y_ctypes),
    model.sigma_obs,
    model.sigma_k,
    model.sigma_m,
    model.sigma,
    model.tau
)

print("Minus log posterior (C++):", mlp_cpp)
print("Minus log posterior (Python):", mpl)

Minus log posterior (C++): 2562662.9550117934
Minus log posterior (Python): 2562662.9550117934


***
## Speed Test

In [4]:
%%timeit -r 100 -n 100

lib.minus_log_posterior(
    params_ctypes, len(params_ctypes),
    t_scaled_ctypes, len(t_scaled_ctypes),
    change_points_ctypes, len(change_points_ctypes),
    model.scale_period,
    normalized_y_ctypes, len(normalized_y_ctypes),
    model.sigma_obs,
    model.sigma_k,
    model.sigma_m,
    model.sigma,
    model.tau
)

285 µs ± 14.5 µs per loop (mean ± std. dev. of 100 runs, 100 loops each)


In [7]:
%%timeit -r 100 -n 100

model._minus_log_posterior(params)

937 µs ± 249 µs per loop (mean ± std. dev. of 100 runs, 100 loops each)


In [8]:
937/285

3.287719298245614